# Lab 09: Monte Carlo Simulation

## Examples (by instructor)

In this lab, you'll practice Monte Carlo simulation from Chapter 14: sampling from input distributions, propagating uncertainty through a model, and summarizing the output distribution to estimate failure rates and reliability.

### Example 1: Estimating π with Monte Carlo

Throw $N$ random darts at a unit square. The fraction landing inside the quarter-circle estimates $\pi/4$.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

rng = np.random.default_rng(seed=0)
N   = 10_000

x = rng.random(N)
y = rng.random(N)
inside = (x**2 + y**2) <= 1.0

pi_est = 4 * np.sum(inside) / N
print(f"N = {N:,}  →  π estimate = {pi_est:.4f}  (error = {abs(pi_est - np.pi):.4f})")

fig, ax = plt.subplots(figsize=(4, 4))
ax.scatter(x[inside],  y[inside],  s=1, color='steelblue', alpha=0.4, label='Inside')
ax.scatter(x[~inside], y[~inside], s=1, color='tomato',    alpha=0.4, label='Outside')
theta = np.linspace(0, np.pi/2, 300)
ax.plot(np.cos(theta), np.sin(theta), 'k-', linewidth=1.5)
ax.set_aspect('equal'); ax.set_title(f'π ≈ {pi_est:.4f}'); ax.legend(markerscale=5, fontsize=9)
plt.tight_layout(); plt.show()

### Example 2: Process Reliability — Propagating Input Uncertainty

A reactor product must have **purity ≥ 97.5%**. Residence time $\tau \sim \mathcal{N}(120, 12^2)$ s and temperature $T \sim \mathcal{N}(400, 8^2)$ K vary each batch. The purity model is:

$$\text{Purity} = 100 \times \left(1 - 0.2\,e^{-k(T)\,\tau}\right), \qquad k(T) = e^{-1550/T}$$

Run 50,000 simulated batches and estimate the failure rate.

In [ ]:
def purity_model(tau, T):
    k = np.exp(-1550.0 / T)
    return 100.0 * (1.0 - 0.2 * np.exp(-k * tau))

tau_mean, tau_std = 120.0, 12.0
T_mean,   T_std   = 400.0,  8.0
spec = 97.5

rng_mc = np.random.default_rng(seed=42)
N_mc   = 50_000

tau_mc = rng_mc.normal(tau_mean, tau_std, N_mc)
T_mc   = rng_mc.normal(T_mean,   T_std,   N_mc)
pur_mc = purity_model(tau_mc, T_mc)

print(f"Mean purity   : {np.mean(pur_mc):.3f} %")
print(f"Std dev       : {np.std(pur_mc, ddof=1):.3f} %")
print(f"Failure rate  : {np.mean(pur_mc < spec)*100:.2f}%  (purity < {spec}%)")

fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(pur_mc, bins=60, color='steelblue', edgecolor='white', density=True, alpha=0.8)
ax.axvline(spec, color='tomato', linestyle='--', linewidth=2, label=f'Spec = {spec}%')
ax.axvline(np.mean(pur_mc), color='black', linestyle='-', linewidth=1.5,
           label=f'Mean = {np.mean(pur_mc):.2f}%')
ax.set_xlabel('Purity (%)'); ax.set_ylabel('Density')
ax.set_title('MC Output Distribution — Product Purity')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

### Example 3: Convergence — How Many Samples Are Enough?

The MC estimate improves as $N$ grows. Error shrinks as $\sim 1/\sqrt{N}$. Track the running failure rate estimate as batches accumulate.

In [ ]:
rng_cv = np.random.default_rng(seed=7)
N_total = 50_000

tau_cv = rng_cv.normal(tau_mean, tau_std, N_total)
T_cv   = rng_cv.normal(T_mean,   T_std,   N_total)
pur_cv = purity_model(tau_cv, T_cv)

# Running failure rate at each step
running_fail = np.cumsum(pur_cv < spec) / np.arange(1, N_total + 1)

fig, ax = plt.subplots(figsize=(8, 4))
ax.semilogx(np.arange(1, N_total + 1), running_fail * 100, color='steelblue', linewidth=1.5)
ax.axhline(running_fail[-1] * 100, color='tomato', linestyle='--', linewidth=1.5,
           label=f'Converged ≈ {running_fail[-1]*100:.2f}%')
ax.set_xlabel('Number of simulated batches (log scale)')
ax.set_ylabel('Estimated failure rate (%)')
ax.set_title('MC Convergence: Running Failure Rate Estimate')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

---

## Warm-Up: Syntax Practice

Short exercises to get comfortable with the MC building blocks before the main problems.

**Exercise 1 — Sample from a distribution.**

Using `np.random.default_rng(seed=10)`, draw 1,000 samples from $\mathcal{N}(\mu=500,\; \sigma=20)$. Print the sample mean and std. Then print the fraction of samples exceeding 530.

In [ ]:
import numpy as np

rng = np.random.default_rng(seed=___)     # seed=10

samples = rng.normal(___, ___, ___)       # mu=500, sigma=20, N=1000

print(f"Sample mean : {np.mean(samples):.2f}")
print(f"Sample std  : {np.std(samples, ddof=1):.2f}")
print(f"Fraction > 530 : {___:.4f}")      # np.mean(samples > 530)

**Exercise 2 — Pass a sample through a model.**

Feed concentration $C_0 \sim \mathcal{N}(2.0,\; 0.1^2)$ mol/L. After a first-order reaction with fixed $k=0.5$ min⁻¹ and $t=3$ min, the outlet concentration is:

$$C = C_0 \cdot e^{-k\,t}$$

Using `seed=5` and $N=5{,}000$ samples, compute the mean and std of $C$. Print what fraction of outlet concentrations fall below 0.25 mol/L.

In [ ]:
import numpy as np

k, t = 0.5, 3.0

rng = np.random.default_rng(seed=___)      # seed=5
C0  = rng.normal(___, ___, ___)            # mu=2.0, sigma=0.1, N=5000

C_out = ___                                # C0 * np.exp(-k * t)

print(f"Mean C_out : {np.mean(C_out):.4f} mol/L")
print(f"Std  C_out : {np.std(C_out, ddof=1):.4f} mol/L")
print(f"Fraction below 0.25 mol/L : {___:.4f}")   # np.mean(C_out < 0.25)

**Exercise 3 — Two uncertain inputs.**

Both $k \sim \mathcal{N}(0.5,\; 0.05^2)$ and $t \sim \mathcal{N}(3.0,\; 0.2^2)$ are uncertain (with fixed $C_0 = 2.0$ mol/L). Using `seed=8` and $N=10{,}000$, sample both, compute $C = C_0\,e^{-k\,t}$, and print mean, std, and fraction below 0.25 mol/L. Compare your answer to Exercise 2 — does uncertainty in $k$ and $t$ increase or decrease the spread?

In [ ]:
import numpy as np

C0 = 2.0

rng = np.random.default_rng(seed=___)         # seed=8
k_s = rng.normal(___, ___, ___)               # mu=0.5, sigma=0.05, N=10000
t_s = rng.normal(___, ___, ___)               # mu=3.0, sigma=0.2,  N=10000

C_out2 = ___                                  # C0 * np.exp(-k_s * t_s)

print(f"Mean C_out : {np.mean(C_out2):.4f} mol/L")
print(f"Std  C_out : {np.std(C_out2, ddof=1):.4f} mol/L")
print(f"Fraction below 0.25 mol/L : {np.mean(C_out2 < 0.25):.4f}")

**Exercise 4 — Summarize output distribution.**

Using the `C_out2` array from Exercise 3, print the 5th and 95th percentiles and plot a histogram with a vertical line at the 5th percentile.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

p5  = np.percentile(___, 5)      # 5th percentile of C_out2
p95 = np.percentile(___, 95)     # 95th percentile

print(f"5th  percentile : {p5:.4f} mol/L")
print(f"95th percentile : {p95:.4f} mol/L")

fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(C_out2, bins=50, color='steelblue', edgecolor='white', alpha=0.8, density=True)
ax.axvline(___, color='tomato', linestyle='--', linewidth=2, label=f'5th pct = {p5:.3f}')
ax.set_xlabel('Outlet concentration (mol/L)')
ax.set_ylabel('Density')
ax.set_title('MC Output Distribution')
ax.legend()
plt.tight_layout()
plt.show()

---

## Practice Problems (by students)

### Problem 1: Heat Exchanger — Outlet Temperature Reliability

A shell-and-tube heat exchanger heats a process stream. The outlet temperature is modeled as:

$$T_\text{out} = T_\text{in} + \frac{U \cdot A \cdot \Delta T_{lm}}{\dot{m} \cdot C_p}$$

For simplicity, use the linearized form:

$$T_\text{out} = T_\text{in} + \eta \cdot (T_\text{hot} - T_\text{in})$$

where $\eta$ is the heat exchanger effectiveness. The following inputs are uncertain each day:

| Variable | Symbol | Distribution |
|----------|--------|-------------|
| Inlet temperature (°C) | $T_\text{in}$ | $\mathcal{N}(25,\; 3^2)$ |
| Hot side temperature (°C) | $T_\text{hot}$ | $\mathcal{N}(120,\; 5^2)$ |
| Effectiveness | $\eta$ | $\mathcal{N}(0.75,\; 0.04^2)$ |

The process requires $T_\text{out} \geq 80°C$.

**(a)** Using `seed=15` and $N = 50{,}000$ samples, simulate $T_\text{out}$ for all three uncertain inputs. Print mean, std, 5th percentile, and the failure rate (% of days below 80°C).

**(b)** Plot the output histogram with vertical lines at the spec limit (80°C) and the mean. Shade the failing region in red.

**(c)** Which input contributes most to variability? Re-run the simulation three times, each time holding **two inputs fixed at their means** and only sampling the third. Print the std of $T_\text{out}$ for each case and identify the most influential input.

### Problem 2: Batch Reactor Yield — Operating Condition Optimization

A batch reactor produces a polymer. The yield (%) is modeled as:

$$Y = 100 \times \left(1 - e^{-k(T)\,\tau}\right), \qquad k(T) = A\,e^{-E_a/(R\,T)}$$

with $A = 2\times10^6$ min⁻¹, $E_a/R = 6{,}000$ K, and the following uncertain operating conditions:

| Variable | Symbol | Baseline | Std dev |
|----------|--------|---------|---------|
| Residence time (min) | $\tau$ | 30 | 3 |
| Temperature (K) | $T$ | 370 | 10 |

The minimum acceptable yield is **85%**.

**(a)** Using `seed=99` and $N=50{,}000$, simulate the yield distribution at the baseline conditions. Print mean, std, and failure rate.

**(b)** The process engineer proposes raising the mean temperature to 380 K (same std = 10 K). Re-run the simulation with this change. By how many percentage points does the failure rate drop?

**(c)** Alternatively, the engineer could **tighten temperature control** to $\sigma_T = 5$ K (keeping mean at 370 K). Re-run this scenario. Compare the failure rate to part (b) — which strategy is more effective?

**(d)** Plot all three yield distributions (baseline, higher $T$, tighter $\sigma_T$) on a single histogram. Add a vertical line at 85%. Which scenario best shifts the distribution away from the spec limit?